# Modelo 5

Nesse notebook, iremos fazer o tratamento restante dos dados, engenharia de features e testar o modelo

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd() / "data"

# Se o notebook estiver em uma subpasta, procura a pasta data acima
if not data_dir.exists():
    for pasta in Path.cwd().parents:
        candidato = pasta / "data"
        if candidato.exists():
            data_dir = candidato
            break

# =========================
# Alarmes
# =========================
alarmes = pd.read_excel(
    data_dir / "Alarmes" / "Histórico de alarmes 23-26.xlsx"
)

# =========================
# Corretiva emergencial
# =========================
emergencial = pd.read_excel(
    data_dir
    / "Corretiva Emergencial"
    / "processados"
    / "ZM01_limpa.xlsx"
)

# =========================
# Corretiva programada
# =========================
programada = pd.read_excel(
    data_dir
    / "Corretiva Programada"
    / "processados"
    / "ZM02_limpa.xlsx"
)

# =========================
# Preventiva
# =========================
preventiva = pd.read_csv(
    data_dir
    / "Preventiva"
    / "preventiva_zm03_limpa.csv"
)

# =========================
# Fluxo / Catracas
# =========================
fluxo = pd.read_excel(
    data_dir
    / "Catracas"
    / "Historico L4.xlsx"
)

In [ ]:
print("Alarmes:", alarmes.shape)
print("Emergencial:", emergencial.shape)
print("Programada:", programada.shape)
print("Preventiva:", preventiva.shape)
print("Fluxo:", fluxo.shape)

In [ ]:
print("\nAlarmes:")
print(alarmes.columns.tolist())

print("\nEmergencial:")
print(emergencial.columns.tolist())

print("\nProgramada:")
print(programada.columns.tolist())

print("\nPreventiva:")
print(preventiva.columns.tolist())

print("\nFluxo:")
print(fluxo.columns.tolist())

In [ ]:
# ==========================================
# PADRONIZAÇÃO DOS NOMES DAS COLUNAS
# ==========================================

# Alarmes PSD
alarmes = alarmes.rename(columns={
    "TAG": "tag_alarme",
    "Horário do Alarme": "horario_alarme",
    "Estação": "estacao",
    "Porta": "porta",
    "Alarme": "tipo_alarme",
    "Horário da Normalizacao": "horario_normalizacao_alarme",
    "Tempo de Normalização": "tempo_normalizacao_alarme"
})


# Corretiva emergencial
emergencial = emergencial.rename(
    columns={
        coluna: f"{coluna}_emergencial"
        for coluna in emergencial.columns
    }
)


# Corretiva programada
programada = programada.rename(
    columns={
        coluna: f"{coluna}_programada"
        for coluna in programada.columns
    }
)


# Preventiva
preventiva = preventiva.rename(columns={
    "Ordem": "ordem_preventiva",
    "Tipo de ordem": "tipo_ordem_preventiva",
    "Centro de trabalho": "centro_trabalho_preventiva",
    "Local de instalação": "local_instalacao_preventiva",
    "Texto breve": "texto_breve_preventiva",
    "Plano de manutenção": "plano_manutencao_preventiva",
    "Status usuário": "status_usuario_preventiva",
    "Status do sistema": "status_sistema_preventiva",
    "Data-base do início": "data_inicio_preventiva",
    "Hr.início programada": "hora_inicio_preventiva",
    "Data-base do fim": "data_fim_preventiva",
    "Fim-base (hora)": "hora_fim_preventiva",
    "DT_Inicio": "inicio_preventiva",
    "DT_Fim": "fim_preventiva",
    "duracao_h": "duracao_h_preventiva",
    "Estacao_Centro": "estacao_centro_preventiva",
    "Estacao_Local": "estacao_local_preventiva",
    "Estacao": "estacao",
    "cancelada": "cancelada_preventiva"
})

In [ ]:
fluxo_bruto = pd.read_excel(
    data_dir / "Catracas" / "Historico L4.xlsx",
    header=None
)

pd.set_option("display.max_columns", None)

display(
    fluxo_bruto.head(10)
)

In [ ]:
# ==========================================
# TRATAMENTO DO FLUXO DE PASSAGEIROS
# ==========================================

# Dados começam na linha 5
fluxo_dados = fluxo_bruto.iloc[5:].copy()

# Data está na coluna 1
fluxo_dados["data"] = pd.to_datetime(
    fluxo_dados[1],
    errors="coerce"
)

# Mapeamento das estações:
# coluna de entrada, coluna de saída
estacoes_fluxo = {
    "BUT": (10, 11),   # Butantã
    "PIN": (12, 13),   # Pinheiros
    "FAL": (14, 15),   # Faria Lima
    "PTA": (16, 17),   # Paulista
    "LUZ": (18, 19),   # Luz
    "FRA": (20, 21),   # Fradique Coutinho
    "HIG": (22, 23),   # Higienópolis-Mackenzie
    "FRE": (24, 25),   # Oscar Freire
    "MBI": (26, 27),   # São Paulo-Morumbi
    "VSO": (28, 29)    # Vila Sônia
}

fluxo_lista = []

for estacao, (col_entrada, col_saida) in estacoes_fluxo.items():

    temp = pd.DataFrame({
        "data": fluxo_dados["data"],
        "estacao": estacao,
        "entrada_fluxo": pd.to_numeric(
            fluxo_dados[col_entrada],
            errors="coerce"
        ),
        "saida_fluxo": pd.to_numeric(
            fluxo_dados[col_saida],
            errors="coerce"
        )
    })

    fluxo_lista.append(temp)

fluxo = pd.concat(
    fluxo_lista,
    ignore_index=True
)

# Remover linhas sem data
fluxo = fluxo.dropna(
    subset=["data"]
)

# Fluxo total da estação no dia
fluxo["total_fluxo"] = (
    fluxo["entrada_fluxo"]
    + fluxo["saida_fluxo"]
)

fluxo = fluxo.sort_values(
    ["data", "estacao"]
).reset_index(drop=True)

display(fluxo.head(20))

In [ ]:
print("Emergencial:")
display(
    emergencial[
        ["local_de_instalacao_emergencial"]
    ]
    .dropna()
    .drop_duplicates()
    .head(30)
)

print("\nProgramada:")
display(
    programada[
        ["local_de_instalacao_programada"]
    ]
    .dropna()
    .drop_duplicates()
    .head(30)
)

In [ ]:
print("Quantidade de locais distintos - Emergencial:")
print(
    emergencial[
        "local_de_instalacao_emergencial"
    ].nunique()
)

print("\nQuantidade de locais distintos - Programada:")
print(
    programada[
        "local_de_instalacao_programada"
    ].nunique()
)

In [ ]:
# ==========================================
# CHAVES DE LOCALIZAÇÃO - EMERGENCIAL
# ==========================================

local_emergencial = (
    emergencial["local_de_instalacao_emergencial"]
    .astype("string")
    .str.upper()
    .str.strip()
)

# Estação: código logo após PSD-
emergencial["estacao"] = local_emergencial.str.extract(
    r"PSD-([A-Z]{3})(?:-|$)",
    expand=False
)

# Plataforma: somente quando explicitamente aparece PLT1 ou PLT2
emergencial["plataforma"] = local_emergencial.str.extract(
    r"-PLT(\d+)",
    expand=False
)

# Porta: PP01, PP02, PP24 etc.
emergencial["porta"] = local_emergencial.str.extract(
    r"-PP(\d+)",
    expand=False
)


# ==========================================
# CHAVES DE LOCALIZAÇÃO - PROGRAMADA
# ==========================================

local_programada = (
    programada["local_de_instalacao_programada"]
    .astype("string")
    .str.upper()
    .str.strip()
)

programada["estacao"] = local_programada.str.extract(
    r"PSD-([A-Z]{3})(?:-|$)",
    expand=False
)

programada["plataforma"] = local_programada.str.extract(
    r"-PLT(\d+)",
    expand=False
)

programada["porta"] = local_programada.str.extract(
    r"-PP(\d+)",
    expand=False
)

In [ ]:
# Plataforma como texto padronizado
emergencial["plataforma"] = (
    emergencial["plataforma"]
    .astype("string")
)

programada["plataforma"] = (
    programada["plataforma"]
    .astype("string")
)

# Porta sempre com dois dígitos
emergencial["porta"] = (
    emergencial["porta"]
    .astype("string")
    .str.zfill(2)
)

programada["porta"] = (
    programada["porta"]
    .astype("string")
    .str.zfill(2)
)

In [ ]:
# ==========================================
# GRANULARIDADE DA LOCALIZAÇÃO
# ==========================================

emergencial["granularidade_localizacao"] = "sem_localizacao"

emergencial.loc[
    emergencial["estacao"].notna(),
    "granularidade_localizacao"
] = "estacao"

emergencial.loc[
    emergencial["plataforma"].notna(),
    "granularidade_localizacao"
] = "plataforma"

emergencial.loc[
    emergencial["porta"].notna(),
    "granularidade_localizacao"
] = "porta"


programada["granularidade_localizacao"] = "sem_localizacao"

programada.loc[
    programada["estacao"].notna(),
    "granularidade_localizacao"
] = "estacao"

programada.loc[
    programada["plataforma"].notna(),
    "granularidade_localizacao"
] = "plataforma"

programada.loc[
    programada["porta"].notna(),
    "granularidade_localizacao"
] = "porta"

In [ ]:
print("EMERGENCIAL")
print(
    emergencial["granularidade_localizacao"]
    .value_counts(dropna=False)
)

print("\nPROGRAMADA")
print(
    programada["granularidade_localizacao"]
    .value_counts(dropna=False)
)

In [ ]:
display(
    emergencial[
        [
            "local_de_instalacao_emergencial",
            "estacao",
            "plataforma",
            "porta",
            "granularidade_localizacao"
        ]
    ].head(30)
)

display(
    programada[
        [
            "local_de_instalacao_programada",
            "estacao",
            "plataforma",
            "porta",
            "granularidade_localizacao"
        ]
    ].head(30)
)

In [ ]:
print("EMERGENCIAL")
print(
    emergencial["granularidade_localizacao"]
    .value_counts(dropna=False)
)

print("\nPROGRAMADA")
print(
    programada["granularidade_localizacao"]
    .value_counts(dropna=False)
)

In [ ]:
print("\nPercentuais - Emergencial")
print(
    emergencial["granularidade_localizacao"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nPercentuais - Programada")
print(
    programada["granularidade_localizacao"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# ==========================================
# EVENTOS ELEGÍVEIS PARA O TARGET
# ==========================================

emergencial["elegivel_target"] = (
    emergencial["granularidade_localizacao"] == "porta"
)

print("Total de manutenções emergenciais:")
print(len(emergencial))

print("\nManutenções emergenciais no nível de porta:")
print(emergencial["elegivel_target"].sum())

print("\nPercentual elegível para o target:")
print(
    round(
        emergencial["elegivel_target"].mean() * 100,
        2
    )
)

In [ ]:
# Garantir datetime
emergencial["inicio_avaria_datetime_emergencial"] = pd.to_datetime(
    emergencial["inicio_avaria_datetime_emergencial"],
    errors="coerce"
)

programada["inicio_avaria_datetime_programada"] = pd.to_datetime(
    programada["inicio_avaria_datetime_programada"],
    errors="coerce"
)

alarmes["horario_alarme"] = pd.to_datetime(
    alarmes["horario_alarme"],
    errors="coerce"
)

preventiva["inicio_preventiva"] = pd.to_datetime(
    preventiva["inicio_preventiva"],
    errors="coerce"
)

print("ALARMES")
print(alarmes["horario_alarme"].min())
print(alarmes["horario_alarme"].max())

print("\nEMERGENCIAL")
print(emergencial["inicio_avaria_datetime_emergencial"].min())
print(emergencial["inicio_avaria_datetime_emergencial"].max())

print("\nPROGRAMADA")
print(programada["inicio_avaria_datetime_programada"].min())
print(programada["inicio_avaria_datetime_programada"].max())

print("\nPREVENTIVA")
print(preventiva["inicio_preventiva"].min())
print(preventiva["inicio_preventiva"].max())

print("\nFLUXO")
print(fluxo["data"].min())
print(fluxo["data"].max())

In [ ]:
# ==========================================
# JANELA TEMPORAL DA MODELAGEM
# ==========================================

inicio_modelagem = pd.Timestamp("2023-02-27")
fim_modelagem = pd.Timestamp("2025-12-31 23:59:59")

horizonte_predicao = pd.Timedelta(days=7)

print("Início da modelagem:", inicio_modelagem)
print("Fim da modelagem:", fim_modelagem)

print(
    "Precisamos de registros emergenciais até:",
    fim_modelagem + horizonte_predicao
)

In [ ]:
# ==========================================
# EXTRAÇÃO GENERALIZADA DA LOCALIZAÇÃO
# DOS ALARMES PSD
# ==========================================

tag = (
    alarmes["tag_alarme"]
    .astype("string")
    .str.upper()
    .str.strip()
)

# ------------------------------------------------
# PADRÃO 1
# Exemplos:
# PSD_FAL_FAL2_PDM09_OBS
# PSD_FAL_FAL2_MDU09_STS
# ------------------------------------------------

padrao_1 = tag.str.extract(
    r"^PSD_([A-Z]{3})_[A-Z]{3}(\d+)_(?:PDM|MDU)(\d+)"
)

# ------------------------------------------------
# PADRÃO 2
# Exemplos:
# PSD_P_FRA2_PDM22_FO
# PSD_P_MBI1_PDM13LSD
# PSD_P_MBI1_MDU22STS
# ------------------------------------------------

padrao_2 = tag.str.extract(
    r"^PSD_P_([A-Z]{3})(\d+)_(?:PDM|MDU)(\d+)"
)

# Combinar os dois padrões
alarmes["estacao_tag"] = (
    padrao_1[0]
    .fillna(padrao_2[0])
)

alarmes["plataforma"] = (
    padrao_1[1]
    .fillna(padrao_2[1])
)

alarmes["porta_tag"] = (
    padrao_1[2]
    .fillna(padrao_2[2])
)

# Padronizar
alarmes["estacao_tag"] = (
    alarmes["estacao_tag"]
    .astype("string")
    .str.upper()
)

alarmes["plataforma"] = (
    alarmes["plataforma"]
    .astype("string")
)

alarmes["porta_tag"] = (
    alarmes["porta_tag"]
    .astype("string")
    .str.zfill(2)
)

alarmes["porta"] = (
    alarmes["porta"]
    .astype("string")
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(2)
)

alarmes["estacao"] = (
    alarmes["estacao"]
    .astype("string")
    .str.upper()
    .str.strip()
)

In [ ]:
print("Total de alarmes:")
print(len(alarmes))

print("\nCom plataforma identificada:")
print(alarmes["plataforma"].notna().sum())

print("\nPercentual com localização:")
print(
    round(
        alarmes["plataforma"].notna().mean() * 100,
        2
    ),
    "%"
)

print("\nConcordância da estação:")
mask_estacao = alarmes["estacao_tag"].notna()

print(
    (
        alarmes.loc[mask_estacao, "estacao"]
        ==
        alarmes.loc[mask_estacao, "estacao_tag"]
    ).mean()
)

print("\nConcordância da porta:")
mask_porta = alarmes["porta_tag"].notna()

print(
    (
        alarmes.loc[mask_porta, "porta"]
        ==
        alarmes.loc[mask_porta, "porta_tag"]
    ).mean()
)

In [ ]:
restantes = alarmes[
    alarmes["plataforma"].isna()
].copy()

restantes["estrutura_tag"] = (
    restantes["tag_alarme"]
    .astype("string")
    .str.upper()
    .str.replace(r"\d+", "<N>", regex=True)
)

print("\nAlarmes ainda não mapeados:")
print(len(restantes))

print("\nEstruturas restantes mais frequentes:")

print(
    restantes["estrutura_tag"]
    .value_counts()
    .head(30)
)

In [ ]:
# ==========================================
# CHAVE ÚNICA DE PORTA
# ==========================================

alarmes["chave_porta"] = (
    alarmes["estacao"].astype("string")
    + "_P" + alarmes["plataforma"].astype("string")
    + "_D" + alarmes["porta"].astype("string")
)

emergencial["chave_porta"] = (
    emergencial["estacao"].astype("string")
    + "_P" + emergencial["plataforma"].astype("string")
    + "_D" + emergencial["porta"].astype("string")
)

programada["chave_porta"] = (
    programada["estacao"].astype("string")
    + "_P" + programada["plataforma"].astype("string")
    + "_D" + programada["porta"].astype("string")
)

In [ ]:
alarmes.loc[
    alarmes[["estacao", "plataforma", "porta"]].isna().any(axis=1),
    "chave_porta"
] = pd.NA

emergencial.loc[
    emergencial[["estacao", "plataforma", "porta"]].isna().any(axis=1),
    "chave_porta"
] = pd.NA

programada.loc[
    programada[["estacao", "plataforma", "porta"]].isna().any(axis=1),
    "chave_porta"
] = pd.NA

In [ ]:
print("Alarmes com chave_porta:")
print(alarmes["chave_porta"].notna().sum())

print("\nEmergenciais com chave_porta:")
print(emergencial["chave_porta"].notna().sum())

print("\nProgramadas com chave_porta:")
print(programada["chave_porta"].notna().sum())

print("\nExemplos:")
display(
    alarmes[
        ["estacao", "plataforma", "porta", "chave_porta"]
    ].dropna().head(10)
)

In [ ]:
# ==========================================
# LOCALIZAÇÃO - MANUTENÇÃO PREVENTIVA
# ==========================================

local_preventiva = (
    preventiva["local_instalacao_preventiva"]
    .astype("string")
    .str.upper()
    .str.strip()
)

# Estação
preventiva["estacao"] = local_preventiva.str.extract(
    r"PSD-([A-Z]{3})(?:-|$)",
    expand=False
)

# Plataforma
preventiva["plataforma"] = local_preventiva.str.extract(
    r"-PLT(\d+)",
    expand=False
)

# Porta, somente se existir explicitamente PPxx
preventiva["porta"] = local_preventiva.str.extract(
    r"-PP(\d+)",
    expand=False
)

# Padronização
preventiva["plataforma"] = (
    preventiva["plataforma"]
    .astype("string")
)

preventiva["porta"] = (
    preventiva["porta"]
    .astype("string")
    .str.zfill(2)
)

In [ ]:
preventiva["granularidade_localizacao"] = "sem_localizacao"

preventiva.loc[
    preventiva["estacao"].notna(),
    "granularidade_localizacao"
] = "estacao"

preventiva.loc[
    preventiva["plataforma"].notna(),
    "granularidade_localizacao"
] = "plataforma"

preventiva.loc[
    preventiva["porta"].notna(),
    "granularidade_localizacao"
] = "porta"

In [ ]:
preventiva["chave_porta"] = (
    preventiva["estacao"].astype("string")
    + "_P" + preventiva["plataforma"].astype("string")
    + "_D" + preventiva["porta"].astype("string")
)

preventiva.loc[
    preventiva[
        ["estacao", "plataforma", "porta"]
    ].isna().any(axis=1),
    "chave_porta"
] = pd.NA

In [ ]:
print("Granularidade da preventiva:")
print(
    preventiva["granularidade_localizacao"]
    .value_counts(dropna=False)
)

print("\nPercentuais:")
print(
    preventiva["granularidade_localizacao"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nPreventivas com chave de porta:")
print(
    preventiva["chave_porta"]
    .notna()
    .sum()
)

In [ ]:
display(
    preventiva[
        [
            "local_instalacao_preventiva",
            "estacao",
            "plataforma",
            "porta",
            "granularidade_localizacao",
            "chave_porta"
        ]
    ].head(30)
)

In [ ]:
# ==========================================
# CHAVES HIERÁRQUICAS DE LOCALIZAÇÃO
# ==========================================

# ---------- ALARMES ----------

alarmes["chave_estacao"] = alarmes["estacao"]

alarmes["chave_plataforma"] = (
    alarmes["estacao"].astype("string")
    + "_P"
    + alarmes["plataforma"].astype("string")
)

alarmes.loc[
    alarmes[["estacao", "plataforma"]].isna().any(axis=1),
    "chave_plataforma"
] = pd.NA


# ---------- EMERGENCIAL ----------

emergencial["chave_estacao"] = emergencial["estacao"]

emergencial["chave_plataforma"] = (
    emergencial["estacao"].astype("string")
    + "_P"
    + emergencial["plataforma"].astype("string")
)

emergencial.loc[
    emergencial[["estacao", "plataforma"]].isna().any(axis=1),
    "chave_plataforma"
] = pd.NA


# ---------- PROGRAMADA ----------

programada["chave_estacao"] = programada["estacao"]

programada["chave_plataforma"] = (
    programada["estacao"].astype("string")
    + "_P"
    + programada["plataforma"].astype("string")
)

programada.loc[
    programada[["estacao", "plataforma"]].isna().any(axis=1),
    "chave_plataforma"
] = pd.NA


# ---------- PREVENTIVA ----------

preventiva["chave_estacao"] = preventiva["estacao"]

preventiva["chave_plataforma"] = (
    preventiva["estacao"].astype("string")
    + "_P"
    + preventiva["plataforma"].astype("string")
)

preventiva.loc[
    preventiva[["estacao", "plataforma"]].isna().any(axis=1),
    "chave_plataforma"
] = pd.NA

In [ ]:
print("ALARMES")
print("Estação:", alarmes["chave_estacao"].notna().sum())
print("Plataforma:", alarmes["chave_plataforma"].notna().sum())
print("Porta:", alarmes["chave_porta"].notna().sum())

print("\nEMERGENCIAL")
print("Estação:", emergencial["chave_estacao"].notna().sum())
print("Plataforma:", emergencial["chave_plataforma"].notna().sum())
print("Porta:", emergencial["chave_porta"].notna().sum())

print("\nPROGRAMADA")
print("Estação:", programada["chave_estacao"].notna().sum())
print("Plataforma:", programada["chave_plataforma"].notna().sum())
print("Porta:", programada["chave_porta"].notna().sum())

print("\nPREVENTIVA")
print("Estação:", preventiva["chave_estacao"].notna().sum())
print("Plataforma:", preventiva["chave_plataforma"].notna().sum())
print("Porta:", preventiva["chave_porta"].notna().sum())

In [ ]:
# ==========================================
# UNIVERSO DE PORTAS
# ==========================================

portas = (
    alarmes[
        ["estacao", "plataforma", "porta", "chave_porta"]
    ]
    .dropna(subset=["chave_porta"])
    .drop_duplicates()
    .sort_values(
        ["estacao", "plataforma", "porta"]
    )
    .reset_index(drop=True)
)

print("Quantidade de portas distintas:")
print(len(portas))

print("\nPortas por estação:")
print(
    portas["estacao"]
    .value_counts()
    .sort_index()
)

display(portas.head(20))

In [ ]:
datas_modelagem = pd.DataFrame({
    "data": pd.date_range(
        start=inicio_modelagem.normalize(),
        end=fim_modelagem.normalize(),
        freq="D"
    )
})

print("Quantidade de dias:")
print(len(datas_modelagem))

print(
    datas_modelagem["data"].min(),
    "→",
    datas_modelagem["data"].max()
)

In [ ]:
# ==========================================
# BASE DIÁRIA PORTA × DATA
# ==========================================

portas["join_key"] = 1
datas_modelagem["join_key"] = 1

base_modelo = (
    portas
    .merge(
        datas_modelagem,
        on="join_key"
    )
    .drop(columns="join_key")
)

portas = portas.drop(columns="join_key")
datas_modelagem = datas_modelagem.drop(columns="join_key")

base_modelo = base_modelo[
    [
        "data",
        "estacao",
        "plataforma",
        "porta",
        "chave_porta"
    ]
].sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

print("Formato da base:")
print(base_modelo.shape)

display(base_modelo.head(20))

In [ ]:
# ==========================================
# EVENTOS EMERGENCIAIS PARA O TARGET
# ==========================================

eventos_emergenciais = (
    emergencial[
        emergencial["elegivel_target"]
        & emergencial["inicio_avaria_datetime_emergencial"].notna()
        & emergencial["chave_porta"].notna()
    ]
    [
        [
            "chave_porta",
            "inicio_avaria_datetime_emergencial"
        ]
    ]
    .drop_duplicates()
    .copy()
)

eventos_emergenciais = eventos_emergenciais.sort_values(
    ["inicio_avaria_datetime_emergencial", "chave_porta"]
).reset_index(drop=True)

print("Eventos emergenciais utilizáveis:")
print(len(eventos_emergenciais))

print("\nPortas diferentes com manutenção emergencial:")
print(eventos_emergenciais["chave_porta"].nunique())

print("\nPeríodo:")
print(eventos_emergenciais["inicio_avaria_datetime_emergencial"].min())
print(eventos_emergenciais["inicio_avaria_datetime_emergencial"].max())

In [ ]:
# ==========================================
# PRÓXIMA MANUTENÇÃO EMERGENCIAL
# ==========================================

base_target = base_modelo.copy()

base_target["data"] = pd.to_datetime(
    base_target["data"]
)

base_target = base_target.sort_values(
    ["data", "chave_porta"]
).reset_index(drop=True)

eventos_emergenciais = eventos_emergenciais.sort_values(
    ["inicio_avaria_datetime_emergencial", "chave_porta"]
).reset_index(drop=True)

base_target = pd.merge_asof(
    base_target,
    eventos_emergenciais,
    left_on="data",
    right_on="inicio_avaria_datetime_emergencial",
    by="chave_porta",
    direction="forward",
    allow_exact_matches=True
)

In [ ]:
base_target["dias_ate_emergencial"] = (
    base_target["inicio_avaria_datetime_emergencial"]
    - base_target["data"]
).dt.total_seconds() / 86400

In [ ]:
# ==========================================
# TARGET: MANUTENÇÃO EMERGENCIAL EM ATÉ 7 DIAS
# ==========================================

base_target["manutencao_emergencial_7d"] = (
    (base_target["dias_ate_emergencial"] >= 0)
    & (base_target["dias_ate_emergencial"] < 7)
).astype(int)

In [ ]:
print("Distribuição do target:")
print(
    base_target["manutencao_emergencial_7d"]
    .value_counts()
)

print("\nPercentual:")
print(
    base_target["manutencao_emergencial_7d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDias até a emergência nos positivos:")
print(
    base_target.loc[
        base_target["manutencao_emergencial_7d"] == 1,
        "dias_ate_emergencial"
    ].describe()
)

In [ ]:
display(
    base_target[
        base_target["manutencao_emergencial_7d"] == 1
    ][
        [
            "data",
            "estacao",
            "plataforma",
            "porta",
            "chave_porta",
            "inicio_avaria_datetime_emergencial",
            "dias_ate_emergencial",
            "manutencao_emergencial_7d"
        ]
    ].head(30)
)

In [ ]:
print("Distribuição de falha_flag na ZM01:")

print(
    emergencial[
        "falha_flag_emergencial"
    ].value_counts(dropna=False)
)

print("\nPercentuais:")

print(
    emergencial[
        "falha_flag_emergencial"
    ]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

In [ ]:
print("ZM01 no nível de porta:")

print(
    emergencial.loc[
        emergencial["elegivel_target"],
        "falha_flag_emergencial"
    ].value_counts(dropna=False)
)

print("\nPercentuais:")

print(
    emergencial.loc[
        emergencial["elegivel_target"],
        "falha_flag_emergencial"
    ]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

In [ ]:
pd.crosstab(
    emergencial["falha_flag_emergencial"],
    emergencial["indisponivel_flag_emergencial"],
    dropna=False,
    margins=True
)

In [ ]:
# ==========================================
# EVENTOS DE FALHA PARA O TARGET
# ==========================================

eventos_falha = (
    emergencial[
        (emergencial["elegivel_target"])
        & (emergencial["falha_flag_emergencial"] == True)
        & (emergencial["inicio_avaria_datetime_emergencial"].notna())
        & (emergencial["chave_porta"].notna())
    ]
    [
        [
            "chave_porta",
            "inicio_avaria_datetime_emergencial"
        ]
    ]
    .drop_duplicates()
    .copy()
)

eventos_falha = eventos_falha.sort_values(
    ["inicio_avaria_datetime_emergencial", "chave_porta"]
).reset_index(drop=True)

print("Eventos de falha utilizáveis:")
print(len(eventos_falha))

print("\nPortas que apresentaram pelo menos uma falha:")
print(eventos_falha["chave_porta"].nunique())

print("\nPeríodo:")
print(eventos_falha["inicio_avaria_datetime_emergencial"].min())
print(eventos_falha["inicio_avaria_datetime_emergencial"].max())

In [ ]:
# ==========================================
# ENCONTRAR A PRÓXIMA FALHA DA PORTA
# ==========================================

base_target = base_modelo.copy()

base_target["data"] = pd.to_datetime(
    base_target["data"]
)

base_target = base_target.sort_values(
    ["data", "chave_porta"]
).reset_index(drop=True)

eventos_falha = eventos_falha.sort_values(
    ["inicio_avaria_datetime_emergencial", "chave_porta"]
).reset_index(drop=True)

base_target = pd.merge_asof(
    base_target,
    eventos_falha,
    left_on="data",
    right_on="inicio_avaria_datetime_emergencial",
    by="chave_porta",
    direction="forward",
    allow_exact_matches=True
)

In [ ]:
base_target["dias_ate_falha"] = (
    base_target["inicio_avaria_datetime_emergencial"]
    - base_target["data"]
).dt.total_seconds() / 86400

In [ ]:
# ==========================================
# TARGET FINAL
# FALHA EMERGENCIAL NOS PRÓXIMOS 7 DIAS
# ==========================================

base_target["falha_emergencial_7d"] = (
    (base_target["dias_ate_falha"] >= 0)
    & (base_target["dias_ate_falha"] < 7)
).astype(int)

In [ ]:
print("Distribuição do target:")
print(
    base_target["falha_emergencial_7d"]
    .value_counts()
)

print("\nPercentual:")
print(
    base_target["falha_emergencial_7d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDias até a falha nos positivos:")
print(
    base_target.loc[
        base_target["falha_emergencial_7d"] == 1,
        "dias_ate_falha"
    ].describe()
)

In [ ]:
# ==========================================
# FEATURES DE HISTÓRICO DE ALARMES
# ==========================================

# Data do alarme
alarmes["data"] = (
    pd.to_datetime(
        alarmes["horario_alarme"],
        errors="coerce"
    )
    .dt.normalize()
)

# Considerar somente alarmes com porta identificada
alarmes_porta = alarmes[
    alarmes["chave_porta"].notna()
    & alarmes["data"].notna()
].copy()

# Quantidade de alarmes por porta e dia
alarmes_diarios = (
    alarmes_porta
    .groupby(
        ["chave_porta", "data"]
    )
    .agg(
        qtd_alarmes_dia=(
            "tipo_alarme",
            "count"
        )
    )
    .reset_index()
)

display(alarmes_diarios.head())

In [ ]:
base_features = base_target.merge(
    alarmes_diarios,
    on=["chave_porta", "data"],
    how="left"
)

# Dias sem alarmes = 0
base_features["qtd_alarmes_dia"] = (
    base_features["qtd_alarmes_dia"]
    .fillna(0)
    .astype(int)
)

base_features = base_features.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

In [ ]:
# ==========================================
# JANELAS TEMPORAIS
# ==========================================

base_features["qtd_alarmes_1d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=1,
            min_periods=1
        ).sum()
    )
)

base_features["qtd_alarmes_3d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=3,
            min_periods=1
        ).sum()
    )
)

base_features["qtd_alarmes_7d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=1
        ).sum()
    )
)

base_features["qtd_alarmes_30d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=30,
            min_periods=1
        ).sum()
    )
)

In [ ]:
display(
    base_features[
        [
            "data",
            "chave_porta",
            "qtd_alarmes_dia",
            "qtd_alarmes_1d",
            "qtd_alarmes_3d",
            "qtd_alarmes_7d",
            "qtd_alarmes_30d",
            "falha_emergencial_7d"
        ]
    ].head(80)
)

In [ ]:
print(
    base_features[
        [
            "qtd_alarmes_1d",
            "qtd_alarmes_3d",
            "qtd_alarmes_7d",
            "qtd_alarmes_30d"
        ]
    ].describe()
)

In [ ]:
# Maiores quantidades de alarmes registrados por uma porta em um dia
top_alarmes_dia = (
    alarmes_diarios
    .sort_values(
        "qtd_alarmes_dia",
        ascending=False
    )
    .head(40)
)

display(top_alarmes_dia)

In [ ]:
# Estatísticas percentis para entender a cauda da distribuição
print(
    alarmes_diarios["qtd_alarmes_dia"]
    .quantile([
        0.90,
        0.95,
        0.99,
        0.995,
        0.999,
        1.0
    ])
)

In [ ]:
top_portas_datas = (
    top_alarmes_dia[
        ["chave_porta", "data"]
    ]
)

detalhes_top = alarmes_porta.merge(
    top_portas_datas,
    on=["chave_porta", "data"],
    how="inner"
)

print(
    detalhes_top[
        [
            "chave_porta",
            "data",
            "tipo_alarme"
        ]
    ]
    .value_counts()
    .head(50)
)

In [ ]:
# ==========================================
# INVESTIGAÇÃO DO MAIOR PICO DE ALARMES
# ==========================================

caso_extremo = alarmes_porta[
    (alarmes_porta["chave_porta"] == "FRA_P1_D03")
    & (alarmes_porta["data"] == pd.Timestamp("2024-05-29"))
].copy()

caso_extremo = caso_extremo.sort_values(
    "horario_alarme"
)

print("Total de registros:")
print(len(caso_extremo))

print("\nTipos de alarme:")
print(
    caso_extremo["tipo_alarme"]
    .value_counts()
)

print("\nPrimeiros registros:")
display(
    caso_extremo[
        [
            "horario_alarme",
            "tipo_alarme",
            "horario_normalizacao_alarme",
            "tag_alarme"
        ]
    ].head(30)
)

In [ ]:
duplicados_exatos = caso_extremo.duplicated(
    subset=[
        "chave_porta",
        "horario_alarme",
        "tipo_alarme"
    ],
    keep=False
)

print("Registros envolvidos em duplicatas exatas:")
print(duplicados_exatos.sum())

print("\nPercentual:")
print(
    round(
        duplicados_exatos.mean() * 100,
        2
    ),
    "%"
)

In [ ]:
caso_extremo["segundos_desde_alarme_anterior"] = (
    caso_extremo["horario_alarme"]
    .diff()
    .dt.total_seconds()
)

print(
    caso_extremo[
        "segundos_desde_alarme_anterior"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.99
        ]
    )
)

In [ ]:
# ==========================================
# VERIFICAÇÃO GLOBAL DE DUPLICATAS
# ==========================================

colunas_evento = [
    "tag_alarme",
    "horario_alarme",
    "tipo_alarme",
    "horario_normalizacao_alarme"
]

duplicados = alarmes.duplicated(
    subset=colunas_evento,
    keep=False
)

print("Total de registros:")
print(len(alarmes))

print("\nRegistros envolvidos em duplicatas:")
print(duplicados.sum())

print("\nPercentual envolvido em duplicatas:")
print(
    round(
        duplicados.mean() * 100,
        2
    ),
    "%"
)

print("\nQuantidade de linhas redundantes:")
print(
    alarmes.duplicated(
        subset=colunas_evento,
        keep="first"
    ).sum()
)

In [ ]:
alarmes_limpos = (
    alarmes
    .drop_duplicates(
        subset=colunas_evento,
        keep="first"
    )
    .copy()
)

print("Antes:", len(alarmes))
print("Depois:", len(alarmes_limpos))
print("Removidos:", len(alarmes) - len(alarmes_limpos))

In [ ]:
caso_extremo_limpo = alarmes_limpos[
    (alarmes_limpos["chave_porta"] == "FRA_P1_D03")
    & (alarmes_limpos["data"] == pd.Timestamp("2024-05-29"))
]

print(
    caso_extremo_limpo["tipo_alarme"]
    .value_counts()
)

print("\nTotal:")
print(len(caso_extremo_limpo))

In [ ]:
alarmes_porta = alarmes_limpos[
    alarmes_limpos["chave_porta"].notna()
    & alarmes_limpos["data"].notna()
].copy()

alarmes_diarios = (
    alarmes_porta
    .groupby(
        ["chave_porta", "data"]
    )
    .agg(
        qtd_alarmes_dia=(
            "tipo_alarme",
            "count"
        )
    )
    .reset_index()
)

In [ ]:
caso_extremo_limpo = (
    caso_extremo_limpo
    .sort_values("horario_alarme")
    .copy()
)

caso_extremo_limpo["segundos_desde_alarme_anterior"] = (
    caso_extremo_limpo["horario_alarme"]
    .diff()
    .dt.total_seconds()
)

print(
    caso_extremo_limpo[
        "segundos_desde_alarme_anterior"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
print("Total de alarmes limpos:")
print(len(caso_extremo_limpo))

print("\nHorários de disparo distintos:")
print(
    caso_extremo_limpo[
        "horario_alarme"
    ].nunique()
)

print("\nQuantidade média de alarmes por instante:")
print(
    len(caso_extremo_limpo)
    /
    caso_extremo_limpo["horario_alarme"].nunique()
)

In [ ]:
# Verificar quantos eventos realmente distintos existem
# considerando TAG + horário de disparo

print("Combinações únicas TAG + horário:")
print(
    caso_extremo_limpo[
        ["tag_alarme", "horario_alarme"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\nQuantidade de registros por TAG + horário:")
print(
    caso_extremo_limpo
    .groupby(
        ["tag_alarme", "horario_alarme"]
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
)

In [ ]:
normalizacoes_por_evento = (
    caso_extremo_limpo
    .groupby(
        ["tag_alarme", "horario_alarme"]
    )
    ["horario_normalizacao_alarme"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    normalizacoes_por_evento.head(30)
)

print("\nEventos com mais de uma normalização:")
print(
    (normalizacoes_por_evento > 1).sum()
)

In [ ]:
# ==========================================
# BASE DE EVENTOS ÚNICOS DE ALARME
# ==========================================

alarmes_eventos = (
    alarmes
    .sort_values("horario_alarme")
    .drop_duplicates(
        subset=[
            "tag_alarme",
            "horario_alarme"
        ],
        keep="first"
    )
    .copy()
)

print("Registros originais:")
print(len(alarmes))

print("\nEventos únicos:")
print(len(alarmes_eventos))

print("\nRegistros redundantes removidos:")
print(
    len(alarmes)
    - len(alarmes_eventos)
)

print("\nPercentual de redução:")
print(
    round(
        (1 - len(alarmes_eventos) / len(alarmes)) * 100,
        2
    ),
    "%"
)

In [ ]:
caso_extremo_eventos = alarmes_eventos[
    (alarmes_eventos["chave_porta"] == "FRA_P1_D03")
    & (
        alarmes_eventos["data"]
        == pd.Timestamp("2024-05-29")
    )
]

print("Eventos únicos no dia:")
print(len(caso_extremo_eventos))

print("\nPor tipo:")
print(
    caso_extremo_eventos[
        "tipo_alarme"
    ].value_counts()
)

In [ ]:
alarmes_porta = alarmes_eventos[
    alarmes_eventos["chave_porta"].notna()
    & alarmes_eventos["data"].notna()
].copy()

alarmes_diarios = (
    alarmes_porta
    .groupby(
        ["chave_porta", "data"]
    )
    .agg(
        qtd_alarmes_dia=(
            "tipo_alarme",
            "count"
        )
    )
    .reset_index()
)

In [ ]:
display(
    alarmes_diarios
    .sort_values(
        "qtd_alarmes_dia",
        ascending=False
    )
    .head(30)
)

In [ ]:
# ==========================================
# RECONSTRUÇÃO DAS FEATURES DE ALARMES
# USANDO EVENTOS ÚNICOS
# ==========================================

alarmes_porta = alarmes_eventos[
    alarmes_eventos["chave_porta"].notna()
    & alarmes_eventos["data"].notna()
].copy()

# Quantidade de eventos únicos por porta e dia
alarmes_diarios = (
    alarmes_porta
    .groupby(["chave_porta", "data"])
    .agg(
        qtd_alarmes_dia=("tipo_alarme", "count")
    )
    .reset_index()
)

# Recomeçar a base de features a partir do target
base_features = base_target.merge(
    alarmes_diarios,
    on=["chave_porta", "data"],
    how="left"
)

base_features["qtd_alarmes_dia"] = (
    base_features["qtd_alarmes_dia"]
    .fillna(0)
    .astype(int)
)

base_features = base_features.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

In [ ]:
base_features["qtd_alarmes_1d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(1, min_periods=1).sum()
    )
)

base_features["qtd_alarmes_3d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).sum()
    )
)

base_features["qtd_alarmes_7d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).sum()
    )
)

base_features["qtd_alarmes_30d"] = (
    base_features
    .groupby("chave_porta")["qtd_alarmes_dia"]
    .transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).sum()
    )
)

In [ ]:
data_inicio_modelo_efetivo = (
    inicio_modelagem.normalize()
    + pd.Timedelta(days=30)
)

base_features = base_features[
    base_features["data"] >= data_inicio_modelo_efetivo
].copy()

base_features = base_features.reset_index(drop=True)

In [ ]:
print(
    base_features[
        [
            "qtd_alarmes_1d",
            "qtd_alarmes_3d",
            "qtd_alarmes_7d",
            "qtd_alarmes_30d"
        ]
    ].describe()
)

print("\nValores ausentes:")
print(
    base_features[
        [
            "qtd_alarmes_1d",
            "qtd_alarmes_3d",
            "qtd_alarmes_7d",
            "qtd_alarmes_30d"
        ]
    ].isna().sum()
)

In [ ]:
print(
    alarmes_eventos["tipo_alarme"]
    .value_counts()
)

In [ ]:
# ==========================================
# QUANTIDADE DIÁRIA POR TIPO DE ALARME
# ==========================================

alarmes_tipo_dia = (
    alarmes_porta
    .groupby(
        [
            "chave_porta",
            "data",
            "tipo_alarme"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(alarmes_tipo_dia.head())

In [ ]:
colunas_tipos = [
    coluna
    for coluna in alarmes_tipo_dia.columns
    if coluna not in ["chave_porta", "data"]
]

alarmes_tipo_dia = alarmes_tipo_dia.rename(
    columns={
        coluna: f"qtd_{coluna.lower()}_dia"
        for coluna in colunas_tipos
    }
)

display(alarmes_tipo_dia.head())

In [ ]:
base_features = base_features.merge(
    alarmes_tipo_dia,
    on=["chave_porta", "data"],
    how="left"
)

colunas_tipo_dia = [
    coluna
    for coluna in base_features.columns
    if coluna.startswith("qtd_")
    and coluna.endswith("_dia")
    and coluna != "qtd_alarmes_dia"
]

base_features[colunas_tipo_dia] = (
    base_features[colunas_tipo_dia]
    .fillna(0)
)

In [ ]:
for coluna in colunas_tipo_dia:

    nome_tipo = (
        coluna
        .replace("qtd_", "")
        .replace("_dia", "")
    )

    base_features[f"qtd_{nome_tipo}_7d"] = (
        base_features
        .groupby("chave_porta")[coluna]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                7,
                min_periods=1
            )
            .sum()
        )
    )

    base_features[f"qtd_{nome_tipo}_30d"] = (
        base_features
        .groupby("chave_porta")[coluna]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                30,
                min_periods=1
            )
            .sum()
        )
    )

In [ ]:
colunas_features_tipo = [
    coluna
    for coluna in base_features.columns
    if (
        coluna.endswith("_7d")
        or coluna.endswith("_30d")
    )
    and coluna.startswith("qtd_")
]

print(
    base_features[
        colunas_features_tipo
    ].describe().T
)

In [ ]:
# ==========================================
# RECONSTRUIR FEATURES POR TIPO DE ALARME
# ANTES DO CORTE DE AQUECIMENTO
# ==========================================

base_features = base_target.merge(
    alarmes_diarios,
    on=["chave_porta", "data"],
    how="left"
)

base_features["qtd_alarmes_dia"] = (
    base_features["qtd_alarmes_dia"]
    .fillna(0)
    .astype(int)
)

base_features = base_features.merge(
    alarmes_tipo_dia,
    on=["chave_porta", "data"],
    how="left"
)

base_features = base_features.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

# Colunas referentes aos tipos de alarme no próprio dia
colunas_tipo_dia = [
    coluna
    for coluna in base_features.columns
    if coluna.startswith("qtd_")
    and coluna.endswith("_dia")
    and coluna != "qtd_alarmes_dia"
]

base_features[colunas_tipo_dia] = (
    base_features[colunas_tipo_dia]
    .fillna(0)
)

In [ ]:
for janela in [1, 3, 7, 30]:

    base_features[f"qtd_alarmes_{janela}d"] = (
        base_features
        .groupby("chave_porta")["qtd_alarmes_dia"]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                janela,
                min_periods=1
            )
            .sum()
        )
    )

In [ ]:
for coluna in colunas_tipo_dia:

    tipo = (
        coluna
        .replace("qtd_", "")
        .replace("_dia", "")
    )

    for janela in [7, 30]:

        base_features[f"qtd_{tipo}_{janela}d"] = (
            base_features
            .groupby("chave_porta")[coluna]
            .transform(
                lambda x: x.shift(1)
                .rolling(
                    janela,
                    min_periods=1
                )
                .sum()
            )
        )

In [ ]:
data_inicio_modelo_efetivo = (
    inicio_modelagem.normalize()
    + pd.Timedelta(days=30)
)

base_features = base_features[
    base_features["data"] >= data_inicio_modelo_efetivo
].copy()

base_features = base_features.reset_index(drop=True)

In [ ]:
colunas_historicas = [
    coluna
    for coluna in base_features.columns
    if (
        coluna.startswith("qtd_")
        and (
            coluna.endswith("_1d")
            or coluna.endswith("_3d")
            or coluna.endswith("_7d")
            or coluna.endswith("_30d")
        )
    )
]

print(
    base_features[
        colunas_historicas
    ].isna().sum()
)

In [ ]:
# ==========================================
# DIAS COM ALGUM ALARME
# ==========================================

base_features["teve_alarme_dia"] = (
    base_features["qtd_alarmes_dia"] > 0
).astype(int)

In [ ]:
for janela in [7, 30]:

    base_features[f"dias_com_alarme_{janela}d"] = (
        base_features
        .groupby("chave_porta")["teve_alarme_dia"]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                janela,
                min_periods=1
            )
            .sum()
        )
    )

In [ ]:
tipos_alarme = [
    coluna
    .replace("qtd_", "")
    .replace("_dia", "")
    for coluna in colunas_tipo_dia
]

tipos_alarme

In [ ]:
colunas_tipos_7d = [
    f"qtd_{tipo}_7d"
    for tipo in tipos_alarme
]

colunas_tipos_30d = [
    f"qtd_{tipo}_30d"
    for tipo in tipos_alarme
]

In [ ]:
base_features["tipos_alarme_7d"] = (
    base_features[colunas_tipos_7d]
    .gt(0)
    .sum(axis=1)
)

base_features["tipos_alarme_30d"] = (
    base_features[colunas_tipos_30d]
    .gt(0)
    .sum(axis=1)
)

In [ ]:
base_features["alarmes_por_dia_ativo_7d"] = (
    base_features["qtd_alarmes_7d"]
    /
    base_features["dias_com_alarme_7d"].replace(0, pd.NA)
)

base_features["alarmes_por_dia_ativo_30d"] = (
    base_features["qtd_alarmes_30d"]
    /
    base_features["dias_com_alarme_30d"].replace(0, pd.NA)
)

In [ ]:
base_features[
    [
        "alarmes_por_dia_ativo_7d",
        "alarmes_por_dia_ativo_30d"
    ]
] = (
    base_features[
        [
            "alarmes_por_dia_ativo_7d",
            "alarmes_por_dia_ativo_30d"
        ]
    ]
    .fillna(0)
    .astype(float)
)

In [ ]:
base_features["media_alarmes_7d"] = (
    base_features["qtd_alarmes_7d"] / 7
)

base_features["media_alarmes_30d"] = (
    base_features["qtd_alarmes_30d"] / 30
)

In [ ]:
base_features["aceleracao_alarmes_7d_30d"] = (
    base_features["media_alarmes_7d"]
    /
    (base_features["media_alarmes_30d"] + 1)
)

In [ ]:
novas_features = [
    "dias_com_alarme_7d",
    "dias_com_alarme_30d",
    "tipos_alarme_7d",
    "tipos_alarme_30d",
    "alarmes_por_dia_ativo_7d",
    "alarmes_por_dia_ativo_30d",
    "media_alarmes_7d",
    "media_alarmes_30d",
    "aceleracao_alarmes_7d_30d"
]

print(
    base_features[
        novas_features
    ].describe().T
)

print("\nValores ausentes:")
print(
    base_features[
        novas_features
    ].isna().sum()
)

In [ ]:
# ==========================================
# CORRIGIR DIAS COM ALARME
# ==========================================

historico_dias_alarme = base_target[
    ["chave_porta", "data"]
].copy()

historico_dias_alarme = historico_dias_alarme.merge(
    alarmes_diarios,
    on=["chave_porta", "data"],
    how="left"
)

historico_dias_alarme["qtd_alarmes_dia"] = (
    historico_dias_alarme["qtd_alarmes_dia"]
    .fillna(0)
)

historico_dias_alarme["teve_alarme_dia"] = (
    historico_dias_alarme["qtd_alarmes_dia"] > 0
).astype(int)

historico_dias_alarme = historico_dias_alarme.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

In [ ]:
for janela in [7, 30]:

    historico_dias_alarme[
        f"dias_com_alarme_{janela}d"
    ] = (
        historico_dias_alarme
        .groupby("chave_porta")["teve_alarme_dia"]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                janela,
                min_periods=1
            )
            .sum()
        )
    )

In [ ]:
historico_dias_alarme = (
    historico_dias_alarme[
        historico_dias_alarme["data"]
        >= data_inicio_modelo_efetivo
    ][
        [
            "chave_porta",
            "data",
            "dias_com_alarme_7d",
            "dias_com_alarme_30d"
        ]
    ]
    .copy()
)

In [ ]:
base_features = base_features.drop(
    columns=[
        "dias_com_alarme_7d",
        "dias_com_alarme_30d"
    ],
    errors="ignore"
)

base_features = base_features.merge(
    historico_dias_alarme,
    on=["chave_porta", "data"],
    how="left"
)

In [ ]:
base_features["alarmes_por_dia_ativo_7d"] = (
    base_features["qtd_alarmes_7d"]
    /
    base_features["dias_com_alarme_7d"].replace(0, pd.NA)
)

base_features["alarmes_por_dia_ativo_30d"] = (
    base_features["qtd_alarmes_30d"]
    /
    base_features["dias_com_alarme_30d"].replace(0, pd.NA)
)

base_features[
    [
        "alarmes_por_dia_ativo_7d",
        "alarmes_por_dia_ativo_30d"
    ]
] = (
    base_features[
        [
            "alarmes_por_dia_ativo_7d",
            "alarmes_por_dia_ativo_30d"
        ]
    ]
    .fillna(0)
    .astype(float)
)

In [ ]:
print(
    base_features[
        [
            "dias_com_alarme_7d",
            "dias_com_alarme_30d",
            "alarmes_por_dia_ativo_7d",
            "alarmes_por_dia_ativo_30d"
        ]
    ].isna().sum()
)

In [ ]:
# ==========================================
# TEMPO DESDE O ÚLTIMO ALARME
# ==========================================

historico_recencia = base_target[
    ["chave_porta", "data"]
].copy()

historico_recencia = historico_recencia.merge(
    alarmes_diarios,
    on=["chave_porta", "data"],
    how="left"
)

historico_recencia["qtd_alarmes_dia"] = (
    historico_recencia["qtd_alarmes_dia"]
    .fillna(0)
)

historico_recencia = historico_recencia.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

In [ ]:
historico_recencia["data_com_alarme"] = (
    historico_recencia["data"]
    .where(
        historico_recencia["qtd_alarmes_dia"] > 0
    )
)

In [ ]:
historico_recencia["data_ultimo_alarme"] = (
    historico_recencia
    .groupby("chave_porta")["data_com_alarme"]
    .transform(
        lambda x: x.shift(1).ffill()
    )
)

In [ ]:
historico_recencia["dias_desde_ultimo_alarme"] = (
    historico_recencia["data"]
    - historico_recencia["data_ultimo_alarme"]
).dt.days

In [ ]:
historico_recencia["sem_alarme_anterior"] = (
    historico_recencia["data_ultimo_alarme"]
    .isna()
    .astype(int)
)

In [ ]:
historico_recencia = historico_recencia[
    historico_recencia["data"] >= data_inicio_modelo_efetivo
][
    [
        "chave_porta",
        "data",
        "dias_desde_ultimo_alarme",
        "sem_alarme_anterior"
    ]
].copy()

In [ ]:
base_features = base_features.merge(
    historico_recencia,
    on=["chave_porta", "data"],
    how="left"
)

In [ ]:
print(
    base_features[
        [
            "dias_desde_ultimo_alarme",
            "sem_alarme_anterior"
        ]
    ].describe()
)

print("\nSem alarme anterior:")
print(
    base_features["sem_alarme_anterior"]
    .value_counts()
)

print("\nValores ausentes:")
print(
    base_features[
        [
            "dias_desde_ultimo_alarme",
            "sem_alarme_anterior"
        ]
    ].isna().sum()
)

In [ ]:
# ==========================================
# HISTÓRICO DE FALHAS ZM01 DA PORTA
# ==========================================

falhas_historicas = (
    eventos_falha[
        [
            "chave_porta",
            "inicio_avaria_datetime_emergencial"
        ]
    ]
    .copy()
)

falhas_historicas["data_falha"] = (
    pd.to_datetime(
        falhas_historicas["inicio_avaria_datetime_emergencial"]
    )
    .dt.normalize()
)

falhas_historicas = (
    falhas_historicas
    .drop_duplicates(
        subset=[
            "chave_porta",
            "inicio_avaria_datetime_emergencial"
        ]
    )
    .sort_values(
        [
            "chave_porta",
            "inicio_avaria_datetime_emergencial"
        ]
    )
    .reset_index(drop=True)
)

print("Eventos históricos de falha:")
print(len(falhas_historicas))

display(falhas_historicas.head())

In [ ]:
print("\nIntervalo temporal:")
print(
    falhas_historicas[
        "inicio_avaria_datetime_emergencial"
    ].min()
)

print(
    falhas_historicas[
        "inicio_avaria_datetime_emergencial"
    ].max()
)

print("\nFalhas por porta:")
print(
    falhas_historicas
    .groupby("chave_porta")
    .size()
    .describe()
)

print("\nPortas com pelo menos uma falha:")
print(
    falhas_historicas[
        "chave_porta"
    ].nunique()
)

In [ ]:
# ==========================================
# FALHAS HISTÓRICAS POR PORTA E DIA
# ==========================================

falhas_dia = (
    falhas_historicas
    .groupby(
        ["chave_porta", "data_falha"]
    )
    .size()
    .reset_index(
        name="qtd_falhas_dia"
    )
)

falhas_dia = falhas_dia.rename(
    columns={
        "data_falha": "data"
    }
)

display(falhas_dia.head())

In [ ]:
historico_falhas = base_target[
    ["chave_porta", "data"]
].copy()

historico_falhas = historico_falhas.merge(
    falhas_dia,
    on=["chave_porta", "data"],
    how="left"
)

historico_falhas["qtd_falhas_dia"] = (
    historico_falhas["qtd_falhas_dia"]
    .fillna(0)
)

historico_falhas = historico_falhas.sort_values(
    ["chave_porta", "data"]
).reset_index(drop=True)

In [ ]:
historico_falhas["qtd_falhas_30d"] = (
    historico_falhas
    .groupby("chave_porta")["qtd_falhas_dia"]
    .transform(
        lambda x: x.shift(1)
        .rolling(
            30,
            min_periods=1
        )
        .sum()
    )
)

historico_falhas["qtd_falhas_90d"] = (
    historico_falhas
    .groupby("chave_porta")["qtd_falhas_dia"]
    .transform(
        lambda x: x.shift(1)
        .rolling(
            90,
            min_periods=1
        )
        .sum()
    )
)

In [ ]:
historico_falhas["qtd_falhas_anteriores"] = (
    historico_falhas
    .groupby("chave_porta")["qtd_falhas_dia"]
    .transform(
        lambda x: x.shift(1)
        .fillna(0)
        .cumsum()
    )
)

In [ ]:
historico_falhas["data_com_falha"] = (
    historico_falhas["data"]
    .where(
        historico_falhas["qtd_falhas_dia"] > 0
    )
)

historico_falhas["data_ultima_falha"] = (
    historico_falhas
    .groupby("chave_porta")["data_com_falha"]
    .transform(
        lambda x: x.shift(1).ffill()
    )
)

historico_falhas["dias_desde_ultima_falha"] = (
    historico_falhas["data"]
    - historico_falhas["data_ultima_falha"]
).dt.days

historico_falhas["sem_falha_anterior"] = (
    historico_falhas["data_ultima_falha"]
    .isna()
    .astype(int)
)

In [ ]:
historico_falhas = historico_falhas[
    historico_falhas["data"]
    >= data_inicio_modelo_efetivo
][
    [
        "chave_porta",
        "data",
        "qtd_falhas_30d",
        "qtd_falhas_90d",
        "qtd_falhas_anteriores",
        "dias_desde_ultima_falha",
        "sem_falha_anterior"
    ]
].copy()

base_features = base_features.merge(
    historico_falhas,
    on=["chave_porta", "data"],
    how="left"
)

In [ ]:
colunas_falha_historica = [
    "qtd_falhas_30d",
    "qtd_falhas_90d",
    "qtd_falhas_anteriores",
    "dias_desde_ultima_falha",
    "sem_falha_anterior"
]

print(
    base_features[
        colunas_falha_historica
    ].describe().T
)

print("\nValores ausentes:")
print(
    base_features[
        colunas_falha_historica
    ].isna().sum()
)

print("\nSem falha anterior:")
print(
    base_features[
        "sem_falha_anterior"
    ].value_counts()
)

In [ ]:
# ==========================================
# PREPARAÇÃO DO HISTÓRICO ZM02
# CORRETIVAS PROGRAMADAS
# ==========================================

eventos_programada = programada[
    programada["chave_porta"].notna()
    & programada["inicio_avaria_datetime_programada"].notna()
].copy()

eventos_programada["data_programada"] = (
    pd.to_datetime(
        eventos_programada["inicio_avaria_datetime_programada"],
        errors="coerce"
    )
    .dt.normalize()
)

eventos_programada = (
    eventos_programada
    .drop_duplicates(
        subset=[
            "chave_porta",
            "inicio_avaria_datetime_programada"
        ]
    )
    .sort_values(
        [
            "chave_porta",
            "inicio_avaria_datetime_programada"
        ]
    )
    .reset_index(drop=True)
)

print("Eventos ZM02 em nível de porta:")
print(len(eventos_programada))

print("\nPortas com pelo menos uma ZM02:")
print(
    eventos_programada["chave_porta"].nunique()
)

print("\nPeríodo:")
print(
    eventos_programada[
        "inicio_avaria_datetime_programada"
    ].min()
)
print(
    eventos_programada[
        "inicio_avaria_datetime_programada"
    ].max()
)

print("\nQuantidade de ZM02 por porta:")
print(
    eventos_programada
    .groupby("chave_porta")
    .size()
    .describe()
)

In [ ]:
colunas_modelo = [
    # Contexto da porta
    "estacao",
    "plataforma",

    # Volume histórico de alarmes
    "qtd_alarmes_1d",
    "qtd_alarmes_3d",
    "qtd_alarmes_7d",
    "qtd_alarmes_30d",

    # Tipos de alarmes recentes
    "qtd_ca_7d",
    "qtd_csd_7d",
    "qtd_fc_7d",
    "qtd_fo_7d",
    "qtd_lsd_7d",
    "qtd_oa_7d",
    "qtd_obs_7d",
    "qtd_rce_7d",
    "qtd_sts_7d",

    # Comportamento recente
    "dias_com_alarme_7d",
    "tipos_alarme_7d",

    # Histórico de falhas
    "qtd_falhas_30d",
    "qtd_falhas_90d",
    "qtd_falhas_anteriores",
    "sem_falha_anterior",

    # Data será usada para o split temporal
    "data",

    # Target
    "falha_emergencial_7d"
]

dataset_modelo = base_features[
    colunas_modelo
].copy()

In [ ]:
print("Shape:")
print(dataset_modelo.shape)

print("\nTipos:")
print(dataset_modelo.dtypes)

print("\nValores ausentes:")
print(dataset_modelo.isna().sum())

print("\nTarget:")
print(
    dataset_modelo["falha_emergencial_7d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
data_corte = pd.Timestamp("2025-07-01")

treino = dataset_modelo[
    dataset_modelo["data"] < data_corte
].copy()

teste = dataset_modelo[
    dataset_modelo["data"] >= data_corte
].copy()

print("TREINO")
print("Período:", treino["data"].min(), "até", treino["data"].max())
print("Linhas:", len(treino))
print("Falhas:", treino["falha_emergencial_7d"].sum())
print(
    "Percentual positivo:",
    round(treino["falha_emergencial_7d"].mean() * 100, 2),
    "%"
)

print("\nTESTE")
print("Período:", teste["data"].min(), "até", teste["data"].max())
print("Linhas:", len(teste))
print("Falhas:", teste["falha_emergencial_7d"].sum())
print(
    "Percentual positivo:",
    round(teste["falha_emergencial_7d"].mean() * 100, 2),
    "%"
)

In [ ]:
data_inicio_teste = pd.Timestamp("2025-07-01")
data_fim_treino = data_inicio_teste - pd.Timedelta(days=7)

treino = dataset_modelo[
    dataset_modelo["data"] <= data_fim_treino
].copy()

teste = dataset_modelo[
    dataset_modelo["data"] >= data_inicio_teste
].copy()

gap = dataset_modelo[
    (dataset_modelo["data"] > data_fim_treino)
    & (dataset_modelo["data"] < data_inicio_teste)
].copy()

print("TREINO")
print(treino["data"].min(), "até", treino["data"].max())
print("Linhas:", len(treino))
print("Positivos:", treino["falha_emergencial_7d"].sum())
print("Taxa:", round(treino["falha_emergencial_7d"].mean() * 100, 2), "%")

print("\nGAP")
print(gap["data"].min(), "até", gap["data"].max())
print("Linhas descartadas:", len(gap))

print("\nTESTE")
print(teste["data"].min(), "até", teste["data"].max())
print("Linhas:", len(teste))
print("Positivos:", teste["falha_emergencial_7d"].sum())
print("Taxa:", round(teste["falha_emergencial_7d"].mean() * 100, 2), "%")

In [ ]:
inicio_validacao = pd.Timestamp("2025-04-01")
fim_treino_selecao = inicio_validacao - pd.Timedelta(days=7)

fim_validacao = pd.Timestamp("2025-06-24")

treino_selecao = dataset_modelo[
    dataset_modelo["data"] <= fim_treino_selecao
].copy()

validacao = dataset_modelo[
    (dataset_modelo["data"] >= inicio_validacao)
    & (dataset_modelo["data"] <= fim_validacao)
].copy()

teste = dataset_modelo[
    dataset_modelo["data"] >= pd.Timestamp("2025-07-01")
].copy()

for nome, df in [
    ("TREINO", treino_selecao),
    ("VALIDAÇÃO", validacao),
    ("TESTE", teste)
]:
    print(f"\n{nome}")
    print("Período:", df["data"].min(), "até", df["data"].max())
    print("Linhas:", len(df))
    print("Falhas:", df["falha_emergencial_7d"].sum())
    print(
        "Positivos:",
        round(df["falha_emergencial_7d"].mean() * 100, 2),
        "%"
    )

In [ ]:
target = "falha_emergencial_7d"

X_treino = treino_selecao.drop(
    columns=["data", target]
)

y_treino = treino_selecao[target]

X_validacao = validacao.drop(
    columns=["data", target]
)

y_validacao = validacao[target]

X_teste = teste.drop(
    columns=["data", target]
)

y_teste = teste[target]

In [ ]:
print("Treino:", X_treino.shape, y_treino.shape)
print("Validação:", X_validacao.shape, y_validacao.shape)
print("Teste:", X_teste.shape, y_teste.shape)

print("\nFeatures:")
print(X_treino.columns.tolist())

In [ ]:
import flaml
from flaml import AutoML
# Cópias para o FLAML
X_treino_flaml = X_treino.copy()
X_validacao_flaml = X_validacao.copy()
X_teste_flaml = X_teste.copy()

# Variáveis categóricas
categoricas = ["estacao", "plataforma"]

for coluna in categoricas:
    X_treino_flaml[coluna] = X_treino_flaml[coluna].astype("category")

    categorias_treino = X_treino_flaml[coluna].cat.categories

    X_validacao_flaml[coluna] = pd.Categorical(
        X_validacao_flaml[coluna],
        categories=categorias_treino
    )

    X_teste_flaml[coluna] = pd.Categorical(
        X_teste_flaml[coluna],
        categories=categorias_treino
    )

In [ ]:
automl = AutoML()

automl.fit(
    X_train=X_treino_flaml,
    y_train=y_treino,

    X_val=X_validacao_flaml,
    y_val=y_validacao,

    task="classification",
    metric="f1",

    time_budget=300,

    eval_method="holdout",
    retrain_full=False,

    seed=42,
    n_jobs=-1,
    verbose=3
)

In [ ]:
print("Melhor algoritmo:")
print(automl.best_estimator)

print("\nMelhores hiperparâmetros:")
print(automl.best_config)

print("\nLoss de validação:")
print(automl.best_loss)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Probabilidade da classe positiva
y_prob_validacao = automl.predict_proba(
    X_validacao_flaml
)[:, 1]

thresholds = [
    0.50,
    0.40,
    0.30,
    0.20,
    0.15,
    0.10,
    0.08,
    0.05,
    0.03,
    0.02,
    0.01
]

resultados_threshold = []

for threshold in thresholds:

    y_pred = (
        y_prob_validacao >= threshold
    ).astype(int)

    resultados_threshold.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_validacao,
            y_pred
        ),
        "precision": precision_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "predicoes_positivas": y_pred.sum()
    })

resultado_threshold = pd.DataFrame(
    resultados_threshold
)

display(
    resultado_threshold.round(4)
)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

pesos_treino = compute_sample_weight(
    class_weight="balanced",
    y=y_treino
)

print("Peso médio classe 0:")
print(pesos_treino[y_treino.values == 0].mean())

print("\nPeso médio classe 1:")
print(pesos_treino[y_treino.values == 1].mean())


In [ ]:
from flaml import AutoML

automl_balanceado = AutoML()

automl_balanceado.fit(
    X_train=X_treino_flaml,
    y_train=y_treino,

    X_val=X_validacao_flaml,
    y_val=y_validacao,

    task="classification",
    metric="f1",

    time_budget=300,

    eval_method="holdout",
    retrain_full=False,

    sample_weight=pesos_treino,

    seed=42,
    n_jobs=-1,
    verbose=3
)

In [137]:
print("Melhor algoritmo:")
print(automl_balanceado.best_estimator)

print("\nMelhores hiperparâmetros:")
print(automl_balanceado.best_config)

print("\nLoss:")
print(automl_balanceado.best_loss)

Melhor algoritmo:
extra_tree

Melhores hiperparâmetros:
{'n_estimators': 24, 'max_features': np.float64(0.12138495209251587), 'max_leaves': 4, 'criterion': np.str_('entropy')}

Loss:
0.9198733560642962


In [138]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred_validacao_balanceado = automl_balanceado.predict(
    X_validacao_flaml
)

print("Accuracy:")
print(round(
    accuracy_score(
        y_validacao,
        y_pred_validacao_balanceado
    ),
    4
))

print("\nPrecision:")
print(round(
    precision_score(
        y_validacao,
        y_pred_validacao_balanceado,
        zero_division=0
    ),
    4
))

print("\nRecall:")
print(round(
    recall_score(
        y_validacao,
        y_pred_validacao_balanceado,
        zero_division=0
    ),
    4
))

print("\nF1:")
print(round(
    f1_score(
        y_validacao,
        y_pred_validacao_balanceado,
        zero_division=0
    ),
    4
))

print("\nMatriz de confusão:")
print(
    confusion_matrix(
        y_validacao,
        y_pred_validacao_balanceado
    )
)

print("\nRelatório:")
print(
    classification_report(
        y_validacao,
        y_pred_validacao_balanceado,
        digits=4,
        zero_division=0
    )
)

Accuracy:
0.8284

Precision:
0.0457

Recall:
0.3225

F1:
0.0801

Matriz de confusão:
[[36147  6863]
 [  691   329]]

Relatório:
              precision    recall  f1-score   support

           0     0.9812    0.8404    0.9054     43010
           1     0.0457    0.3225    0.0801      1020

    accuracy                         0.8284     44030
   macro avg     0.5135    0.5815    0.4928     44030
weighted avg     0.9596    0.8284    0.8863     44030



In [139]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

y_prob_validacao_balanceado = (
    automl_balanceado.predict_proba(
        X_validacao_flaml
    )[:, 1]
)

thresholds = [
    0.50,
    0.45,
    0.40,
    0.35,
    0.30,
    0.25,
    0.20,
    0.15,
    0.10
]

resultados_threshold_balanceado = []

for threshold in thresholds:

    y_pred = (
        y_prob_validacao_balanceado >= threshold
    ).astype(int)

    resultados_threshold_balanceado.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_validacao,
            y_pred
        ),
        "precision": precision_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validacao,
            y_pred,
            zero_division=0
        ),
        "predicoes_positivas": y_pred.sum()
    })

resultado_threshold_balanceado = pd.DataFrame(
    resultados_threshold_balanceado
)

display(
    resultado_threshold_balanceado.round(4)
)

,threshold,accuracy,precision,recall,f1,predicoes_positivas
0,0.50,0.8284,0.0457,0.3225,0.0801,7192
1,0.45,0.0233,0.0232,1.0000,0.0453,44025
2,0.40,0.0232,0.0232,1.0000,0.0453,44030
3,0.35,0.0232,0.0232,1.0000,0.0453,44030
4,0.30,0.0232,0.0232,1.0000,0.0453,44030
5,0.25,0.0232,0.0232,1.0000,0.0453,44030
6,0.20,0.0232,0.0232,1.0000,0.0453,44030
7,0.15,0.0232,0.0232,1.0000,0.0453,44030
8,0.10,0.0232,0.0232,1.0000,0.0453,44030
